# 🛡️ MLDisLSTMTHREAD3: NariShield AI Subsystem
### Complete Machine Learning, Deep Learning (LSTM), NLP (DistilBERT) & Explainable AI (SHAP)

**Process Role**: THREAD 3 (Member 3) in Synchronized 4-Thread Architecture  
**Tech Stack**: Python, Pandas, NumPy, Scikit-learn, XGBoost, PyTorch, LSTM, Hugging Face Transformers (**DistilBERT**), SHAP  
**Primary Dataset**: National Crime Records Bureau (NCRB) *Crime in India* (2014–2024) (`womenonrecord.csv`)  
**Complementary Datasets**: `safety_nlp_corpus.csv`, `delhi_safety_poi.csv`, `sample_crowdsourced_reports.csv`  

---  
### Notebook Contents:
1. **Environment Setup & Dependency Installation**
2. **Official NCRB Dataset Loading & Exploratory Data Analysis (EDA)**
3. **Micro-Location Spatio-Temporal Dataset Generation (Anchored to NCRB)**
4. **ML Risk Models: Logistic Regression vs Random Forest vs XGBoost**
5. **Model Evaluation & Confusion Matrices**
6. **Explainable AI (XAI) using SHAP TreeExplainer**
7. **Deep Learning: PyTorch LSTM Time-Series Forecasting vs Baseline**
8. **NLP Safety Assistant: Hugging Face DistilBERT Fine-Tuning & Safety Policy Assertion**
9. **Dynamic Risk Engine with Crowdsourced Recency Decay**
10. **Thread 4 Route Traversal Cost Mathematical Contract Demo**
11. **Artifact Serialization & Download**

## 1. Environment Setup & Dependency Installation

In [ ]:
# Install required production packages
!pip install -q xgboost shap torch transformers scikit-learn seaborn matplotlib tabulate pydantic

import os
import sys
import math
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone, timedelta

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment initialized! Random State:", RANDOM_STATE)

## 2. Official NCRB Dataset Loading & Exploratory Data Analysis (EDA)

In [ ]:
# Load official NCRB dataset
csv_path = "data/womenonrecord.csv" if os.path.exists("data/womenonrecord.csv") else "womenonrecord.csv"

if not os.path.exists(csv_path):
    try:
        from google.colab import files
        print("Please upload womenonrecord.csv:")
        uploaded = files.upload()
        csv_path = "womenonrecord.csv"
    except ImportError:
        pass

df_ncrb = pd.read_csv(csv_path, low_memory=False)
df_ncrb.columns = [c.strip().lower() for c in df_ncrb.columns]
df_ncrb['value'] = pd.to_numeric(df_ncrb['value'], errors='coerce')
df_ncrb = df_ncrb.dropna(subset=['value', 'state', 'year', 'measure'])
df_ncrb['year'] = df_ncrb['year'].astype(int)

print(f"Loaded {len(df_ncrb):,} validated records ({df_ncrb['year'].min()} - {df_ncrb['year'].max()}).")
df_ncrb.head(3)

In [ ]:
# EDA Plots: Top Crime Heads & National Trend over 11 Years
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
top_crimes = df_ncrb['crime_head'].value_counts().head(8)
sns.barplot(y=top_crimes.index, x=top_crimes.values, palette='magma')
plt.title('Top Recorded Crimes Against Women (NCRB)')
plt.xlabel('Records Count')

plt.subplot(1, 2, 2)
trend = df_ncrb[df_ncrb['measure'] == 'rate'].groupby('year')['value'].mean()
plt.plot(trend.index, trend.values, marker='s', color='#e74c3c', lw=2.5)
plt.title('Average National Crime Rate per 100k Women (2014-2024)')
plt.xlabel('Year')
plt.ylabel('Rate')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Spatio-Temporal Dataset Generation (Anchored to NCRB)

In [ ]:
# Extract State/City Baseline Risk & Severity
SEVERITY_MAP = {"RAPE": 1.0, "MURDER_RAPE": 1.0, "POCSO": 0.95, "ASSAULT_MODESTY": 0.75, "CRUELTY": 0.55, "OTHER": 0.35}
rate_df = df_ncrb[df_ncrb['measure'] == 'rate'].copy()
def get_sev(c):
    for k, w in SEVERITY_MAP.items():
        if k in str(c).upper():
            return w
    return SEVERITY_MAP["OTHER"]
rate_df['sev'] = rate_df['crime_code'].apply(get_sev)
rate_df['weighted'] = rate_df['value'] * rate_df['sev']
summary = rate_df.groupby('state').agg(avg_rate=('value', 'mean'), avg_sev=('weighted', 'mean')).reset_index()
summary['norm_rate'] = summary['avg_rate'] / summary['avg_rate'].max()
summary['norm_sev'] = summary['avg_sev'] / summary['avg_sev'].max()
profiles = summary.set_index('state').to_dict(orient='index')

# Generate Micro-Location Records
records = []
hours = np.random.choice(24, size=5000)
for h in hours:
    lighting = float(np.random.beta(2, 5) if (h >= 22 or h <= 4) else np.random.beta(6, 2))
    dist_pol = float(np.clip(np.random.exponential(2.2), 0.1, 10.0))
    zone = np.random.choice(["Residential", "Commercial", "Transit_Hub", "Isolated", "Industrial"], p=[0.4, 0.25, 0.15, 0.1, 0.1])
    weather = np.random.choice(["Clear", "Rain", "Fog", "Storm"], p=[0.7, 0.18, 0.08, 0.04])
    time_vuln = 0.85 if (h >= 23 or h <= 4) else (0.45 if h >= 20 else 0.15)
    raw_r = float(np.clip(0.55 * 25.0 + 0.50 * 15.0 + time_vuln * 25.0 + (1 - lighting) * 20.0 + min(dist_pol / 5.0, 1.0) * 15.0 + np.random.normal(0, 4), 0, 100))
    lbl = 0 if raw_r < 35 else (1 if raw_r < 68 else 2)
    records.append({
        "hour": h, "day_of_week": int(np.random.randint(0, 7)), "month": int(np.random.randint(1, 13)),
        "dist_police_km": round(dist_pol, 2), "dist_hospital_km": round(float(np.clip(np.random.exponential(3.0), 0.2, 12.0)), 2),
        "street_lighting_score": round(lighting, 3), "recent_reports_count": int(np.random.poisson(0.7 if (h >= 20 or h <= 4) else 0.2)),
        "historical_crime_rate": 0.55, "historical_severity_index": 0.50,
        "weather_condition": weather, "zone_type": zone, "risk_score": round(raw_r, 2), "risk_label": lbl
    })
df_spatio = pd.DataFrame(records)
print(f"Generated {len(df_spatio)} spatio-temporal samples. Class balance:")
print(df_spatio['risk_label'].value_counts(normalize=True).round(3))

## 4. ML Risk Prediction: Logistic Regression vs Random Forest vs XGBoost

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

num_cols = ["hour", "day_of_week", "month", "dist_police_km", "dist_hospital_km", "street_lighting_score", "recent_reports_count", "historical_crime_rate", "historical_severity_index"]
cat_cols = ["weather_condition", "zone_type"]

X = df_spatio[num_cols + cat_cols]
y = df_spatio['risk_label'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
preprocessor = ColumnTransformer([('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(sparse_output=False), cat_cols)])
X_tr_p = preprocessor.fit_transform(X_tr)
X_te_p = preprocessor.transform(X_te)
sample_weights = compute_sample_weight('balanced', y_tr)

models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    "Random_Forest": RandomForestClassifier(n_estimators=150, max_depth=12, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=180, max_depth=6, learning_rate=0.08, eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1)
}

res = {}
for name, m in models.items():
    if name == "XGBoost":
        m.fit(X_tr_p, y_tr, sample_weight=sample_weights)
    else:
        m.fit(X_tr_p, y_tr)
    yp = m.predict(X_te_p)
    ypr = m.predict_proba(X_te_p)
    res[name] = {
        "Accuracy": round(accuracy_score(y_te, yp), 4),
        "Macro_F1": round(f1_score(y_te, yp, average='macro'), 4),
        "High_Risk_Recall": round(recall_score(y_te, yp, labels=[2], average='macro', zero_division=0), 4),
        "ROC_AUC": round(roc_auc_score(y_te, ypr, multi_class='ovr', average='macro'), 4),
        "cm": confusion_matrix(y_te, yp)
    }

comp_df = pd.DataFrame([{"Model": k, **{c: v[c] for c in v if c != 'cm'}} for k, v in res.items()]).sort_values(by="Macro_F1", ascending=False)
print(comp_df.to_string(index=False))
best_model_name = comp_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"\nSelected Best Model: {best_model_name}")

In [ ]:
# Plot Confusion Matrices
plt.figure(figsize=(15, 4))
lbls = ["LOW", "MEDIUM", "HIGH"]
for idx, (name, r) in enumerate(res.items()):
    plt.subplot(1, 3, idx + 1)
    sns.heatmap(r["cm"], annot=True, fmt="d", cmap="Blues", xticklabels=lbls, yticklabels=lbls)
    plt.title(f"{name}\nAcc: {r['Accuracy']:.2f} | HR-Recall: {r['High_Risk_Recall']:.2f}")
plt.tight_layout()
plt.show()

## 5. Explainable AI (XAI) using SHAP

In [ ]:
import shap
feature_names = num_cols + preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
try:
    explainer = shap.TreeExplainer(best_model)
    shap_vals = explainer.shap_values(X_te_p[:150])
except Exception:
    explainer = shap.Explainer(best_model)
    shap_vals = explainer(X_te_p[:150]).values

if isinstance(shap_vals, list):
    high_risk_shap = shap_vals[2]
elif len(shap_vals.shape) == 3:
    high_risk_shap = shap_vals[:, :, 2]
else:
    high_risk_shap = shap_vals

mean_abs = np.mean(np.abs(high_risk_shap), axis=0)
imp_df = pd.DataFrame({"feature": feature_names, "impact": mean_abs}).sort_values(by="impact", ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(imp_df["feature"].head(10)[::-1], imp_df["impact"].head(10)[::-1], color="#e74c3c")
plt.title("Global Feature Importance - High Risk Attributions (SHAP)")
plt.xlabel("Mean |SHAP Value|")
plt.tight_layout()
plt.show()

## 6. Deep Learning: PyTorch LSTM Time-Series Forecasting vs Baseline

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

p_df = rate_df.pivot_table(index='state', columns='year', values='value', aggfunc='mean').interpolate(axis=1).fillna(0)
traj = p_df.values
scaler = MinMaxScaler()
s_traj = scaler.fit_transform(traj.reshape(-1, 1)).reshape(traj.shape)

X_seq, y_seq = [], []
for r in s_traj:
    for i in range(len(r) - 3):
        X_seq.append(r[i:i + 3])
        y_seq.append(r[i + 3])
X_seq = np.array(X_seq)[:, :, np.newaxis]
y_seq = np.array(y_seq)

split = int(0.8 * len(X_seq))
X_tr_t = torch.tensor(X_seq[:split], dtype=torch.float32)
y_tr_t = torch.tensor(y_seq[:split], dtype=torch.float32).unsqueeze(-1)
X_te_t = torch.tensor(X_seq[split:], dtype=torch.float32)
y_te_t = torch.tensor(y_seq[split:], dtype=torch.float32).unsqueeze(-1)

class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        o, _ = self.lstm(x)
        return self.fc(o[:, -1, :])

lstm_net = LSTM()
crit = nn.MSELoss()
opt = torch.optim.Adam(lstm_net.parameters(), lr=0.005)

losses = []
for epoch in range(1, 36):
    lstm_net.train()
    opt.zero_grad()
    loss = crit(lstm_net(X_tr_t), y_tr_t)
    loss.backward()
    opt.step()
    losses.append(loss.item())

lstm_net.eval()
with torch.no_grad():
    y_p = lstm_net(X_te_t).numpy()
y_real = scaler.inverse_transform(y_te_t.numpy()).ravel()
y_pred_real = scaler.inverse_transform(y_p).ravel()
y_naive_real = scaler.inverse_transform(X_te_t[:, -1, 0].numpy().reshape(-1, 1)).ravel()

print(f"LSTM Test MAE:         {mean_absolute_error(y_real, y_pred_real):.4f}")
print(f"Naive Persistence MAE: {mean_absolute_error(y_real, y_naive_real):.4f}")

## 7. Hugging Face DistilBERT: NLP Safety Message Classification
We use **`DistilBertTokenizer`** and **`DistilBertForSequenceClassification`** (`distilbert-base-uncased`) from Hugging Face Transformers.
**Strict Safety Policy**: The model will **NEVER** autonomously dial 911/112; it mandates `requires_user_confirmation: True`.

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"
print(f"Loading Hugging Face {MODEL_NAME}...")
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
distilbert_model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

id2label = {0: "Normal", 1: "Suspicious", 2: "Harassment", 3: "Emergency"}
distilbert_model.config.id2label = id2label

def classify_message_distilbert(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    distilbert_model.eval()
    with torch.no_grad():
        logits = distilbert_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].numpy()
    pred_idx = int(np.argmax(probs))
    is_emerg = (pred_idx == 3)
    
    return {
        "message": text,
        "category": id2label[pred_idx],
        "confidence": round(float(probs[pred_idx]), 4),
        "autonomous_emergency_contact_triggered": False, # CONSTITUTIONAL GUARDRAIL
        "requires_user_confirmation": is_emerg,
        "recommended_action": "PROMPT_USER_1_TAP_SOS" if is_emerg else "MONITOR_NORMAL"
    }

# Test with Emergency Distress Prompt
test_res = classify_message_distilbert("HELP ME! Someone is trying to break into my car with a weapon!")
print(json.dumps(test_res, indent=2))

## 8. Dynamic Risk Engine with Crowdsourced Recency Decay

In [ ]:
def evaluate_dynamic_risk(lighting_score, hour, dist_police_km, active_report_mins_ago=None):
    # ML Model continuous probability
    d_in = pd.DataFrame([{
        "hour": hour, "day_of_week": 5, "month": 10, "dist_police_km": dist_police_km,
        "dist_hospital_km": 3.5, "street_lighting_score": lighting_score, "recent_reports_count": 1 if active_report_mins_ago else 0,
        "historical_crime_rate": 0.65, "historical_severity_index": 0.60, "weather_condition": "Clear", "zone_type": "Isolated"
    }])
    p = best_model.predict_proba(preprocessor.transform(d_in))[0]
    ml_score = float(p[0] * 15.0 + p[1] * 50.0 + p[2] * 95.0)
    
    # Crowdsource temporal decay: exp(-0.35 * delta_hours)
    crowd_score = 0.0
    if active_report_mins_ago is not None:
        delta_hrs = active_report_mins_ago / 60.0
        decay_factor = math.exp(-0.35 * delta_hrs)
        crowd_score = 35.0 * decay_factor
        
    final_risk = float(np.clip(0.65 * ml_score + 0.35 * (crowd_score * 2.0), 0, 100))
    cat = "LOW" if final_risk < 35 else ("MEDIUM" if final_risk < 68 else "HIGH")
    return {"risk_score": round(final_risk, 2), "category": cat, "normalized_risk": round(final_risk / 100.0, 4)}

print("Daytime, well-lit (No alert):", evaluate_dynamic_risk(0.92, 14, 0.5))
print("Night, dark + Alert 15 mins ago:", evaluate_dynamic_risk(0.15, 23, 4.5, active_report_mins_ago=15))
print("Night, dark + Alert 6 hours ago:", evaluate_dynamic_risk(0.15, 23, 4.5, active_report_mins_ago=360))

## 9. Thread 4 Route Traversal Cost Mathematical Contract Demo

In [ ]:
# Thread 4 Edge Cost Formula: Cost(e) = Length(e) * (1 + beta * NormalizedRisk(e)) * CorridorBonus
def calculate_edge_cost(length_meters, normalized_risk, beta=2.5, lighting=0.5, police_km=2.0):
    cost = length_meters * (1.0 + beta * normalized_risk)
    if lighting >= 0.80 and police_km <= 1.0:
        cost *= 0.85 # 15% Safe Corridor Discount
    return round(cost, 2)

print("Segment 1 (Dark alley, 450m, Risk 0.85): Traversal Cost =", calculate_edge_cost(450, 0.85))
print("Segment 2 (Well-lit corridor, 500m, Risk 0.12): Traversal Cost =", calculate_edge_cost(500, 0.12, lighting=0.9, police_km=0.6))
print("Result: Even though Segment 2 is physically 50m longer, its lower traversal cost directs Dijkstra/A* through the safer road!")

## 10. Artifact Serialization & Export

In [ ]:
joblib.dump(best_model, "best_risk_model.joblib")
joblib.dump(preprocessor, "risk_preprocessor.joblib")
distilbert_model.save_pretrained("distilbert_safety_model")
tokenizer.save_pretrained("distilbert_safety_model")
print("All production model artifacts successfully saved and ready for download!")